# SMS Guard — запуск в Google Colab через туннель

Блокнот скачивает проект, ставит зависимости, обучает SMS и URL модели, прогоняет тесты,
создаёт демо-пользователей и запускает Django-сайт через `cloudflared`-туннель.

## 1. Получить проект (выполните ОДИН из вариантов)

**Вариант А — из GitHub:**

In [ ]:
import os
if not os.path.exists("/content/project_AI"):
    !git clone https://github.com/ruslanalimov800-max/project_AI.git
%cd /content/project_AI
!git pull

**Вариант Б — загрузить zip-архив проекта** (если репозиторий приватный). Папку `.git` и `.venv` в архив класть не нужно:

In [ ]:
from google.colab import files
import zipfile, glob, os

name = next(iter(files.upload()))
zipfile.ZipFile(name).extractall("/content/project")
root = os.path.dirname(glob.glob("/content/project/**/manage.py", recursive=True)[0])
os.chdir(root)
print("Готово:", os.getcwd())

## 2. Установить зависимости

pandas, NumPy и scikit-learn в Colab уже есть, ставим недостающие пакеты.

In [ ]:
!pip install -q django djangorestframework python-dotenv pyyaml tldextract pyarrow pytest pytest-django

## 3. Данные и обучение моделей (SMS + URL)

Обучаем модели в среде Colab, чтобы `.joblib` соответствовали версиям библиотек Colab.

In [ ]:
!python -m ml.data.prepare_sms
!python -m ml.sms_compare
!python -m ml.url_model

## 4. Тесты

In [ ]:
!python -m pytest -q

## 5. Метрики и графики

In [ ]:
import json
from IPython.display import Image, display

sms_meta = json.load(open("ml/artifacts/sms_model_meta.json", encoding="utf-8"))
url_meta = json.load(open("ml/artifacts/url_model_meta.json", encoding="utf-8"))
print("SMS модель:", sms_meta.get("algorithm"), sms_meta.get("metrics"))
print("URL модель:", url_meta.get("algorithm"), url_meta.get("metrics"))
for name in ["sms_models_roc_pr", "url_models_roc", "url_feature_importance_top10"]:
    display(Image(f"docs/figures/{name}.png"))

## 6. Проверка анализатора на примерах

In [ ]:
import sys, os
sys.path.insert(0, os.getcwd())
from ml.analyzer import analyze_message

for text in [
    "Срочно переведите 50000 тенге и сообщите код из SMS! http://kaspi1.kz/auth",
    "Привет, встретимся завтра в 18:00 у входа",
    "Сәлем! Қалың қалай?",
]:
    res = analyze_message(text)
    print(res["category"], res["risk_score"], "<-", text)

## 7. Запустить сайт (Django + Cloudflare Tunnel)

После запуска откройте выведенную ссылку `https://....trycloudflare.com`. Логин демо-пользователя: `demo_user` / `DemoPass123`.

In [ ]:
%%writefile .env
DEBUG=1
ALLOWED_HOSTS=*
CSRF_TRUSTED_ORIGINS=https://*.trycloudflare.com

In [ ]:
!python manage.py migrate
!python manage.py seed_demo
!python manage.py check

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

import re, subprocess, time, urllib.request

subprocess.Popen(["python", "manage.py", "runserver", "0.0.0.0:8000", "--noreload"],
                 stdout=open("/tmp/django.log", "w"), stderr=subprocess.STDOUT)
time.sleep(5)
print("Health внутри Colab:", urllib.request.urlopen("http://127.0.0.1:8000/api/v1/health/").read().decode())

tunnel = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in tunnel.stdout:
    m = re.search(r"https://[a-zA-Z0-9.-]+\.trycloudflare\.com", line)
    if m:
        print("\n>>> ССЫЛКА НА САЙТ:", m.group(0))
        print(">>> Демо-вход: demo_user / DemoPass123")
        print("Чтобы остановить сервер, нажмите Stop у этой ячейки.\n")
        break
tunnel.wait()